# MLflow Regression Recipe Notebook


In [ ]:
import pandas as pd
import logging
import sklearn
from sklearn.model_selection import train_test_split
from pandas import DataFrame
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [ ]:
DATA_LOCATION = "../data/nyc.parquet"

In [ ]:
df = pd.read_parquet(DATA_LOCATION, index_col=0)

In [ ]:
def create_dataset_filter(dataset: pd.DataFrame) -> pd.Series[bool]:
    """
    Mark rows of the split datasets to be additionally filtered. This function will be called on
    the training, validation, and test datasets.
    :param dataset: The {train,validation,test} dataset produced by the data splitting procedure.
    :return: A Series indicating whether each row should be filtered
    """

    return (
        (dataset["fare_amount"] > 0)
        & (dataset["trip_distance"] < 400)
        & (dataset["trip_distance"] > 0)
        & (dataset["fare_amount"] < 1000)
    ) | (~dataset.isna().any(axis=1))

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df, test_size=0.25, random_state=42)

In [ ]:
df_train, df_val, df_test = (
    create_dataset_filter(df_train),
    create_dataset_filter(df_val),
    create_dataset_filter(df_test),
)

In [ ]:
def calculate_features(df: DataFrame):
    df["pickup_dow"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    trip_duration = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    df["trip_duration"] = trip_duration.map(lambda x: x.total_seconds() / 60)
    dateTimeColumns = list(df.select_dtypes(include=["datetime64"]).columns)
    df[dateTimeColumns] = df[dateTimeColumns].astype(str)
    df.drop(columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"], inplace=True)
    return df

In [ ]:
function_transformer_params = (
    {} if sklearn.__version__.startswith("1.0") else {"feature_names_out": "one-to-one"}
)

In [ ]:
p = Pipeline(
    steps=[
        (
            "calculate_time_and_duration_features",
            FunctionTransformer(calculate_features, **function_transformer_params),
        ),
        (
            "encoder",
            ColumnTransformer(
                transformers=[
                    (
                        "hour_encoder",
                        OneHotEncoder(categories="auto", sparse=False),
                        ["pickup_hour"],
                    ),
                    (
                        "day_encoder",
                        OneHotEncoder(categories="auto", sparse=False),
                        ["pickup_dow"],
                    ),
                    (
                        "std_scaler",
                        StandardScaler(),
                        ["trip_distance", "trip_duration"],
                    ),
                ]
            ),
        ),
    ]
)

In [ ]:
df_train = p.fit(df_train)

In [ ]:
from sklearn.linear_model import SGDRegressor

estimator_params = {}

model = SGDRegressor(random_state=42, **estimator_params)

In [ ]:
model.fit(df_train.drop(columns=["fare_amount"]), df_train["fare_amount"])

In [ ]:
nyc_output = model.predict(df_val.drop(columns=["fare_amount"]))
nyc_output.to_parquet("../data/nyc_output.parquet")